In [ ]:
import networkx as nx
from typing import List, Dict, Any, Optional
from collections import defaultdict
import json

class GraphQueryEngine:
    """Engine for performing traversal-based queries on the knowledge graph."""
    
    def __init__(self, graph: nx.DiGraph):
        self.graph = graph
    
    def find_documents_with_entity(self, entity_type: str, entity_text: str) -> List[Dict[str, Any]]:
        """Find all documents containing a specific entity."""
        entity_id = f"{entity_type}_{entity_text}"
        if not self.graph.has_node(entity_id):
            return []
        
        # Find documents containing this entity (predecessors because CONTAINS edge goes doc→entity)
        doc_nodes = list(self.graph.predecessors(entity_id))
        
        return [
            {
                'document_id': doc_id,
                'content': self.graph.nodes[doc_id]['content'],
                'metadata': self.graph.nodes[doc_id]['metadata']
            }
            for doc_id in doc_nodes
        ]
    
    def find_related_entities(self, entity_type: str, entity_text: str, 
                            min_weight: int = 1) -> List[Dict[str, Any]]:
        """Find entities related to a given entity through co-occurrence."""
        entity_id = f"{entity_type}_{entity_text}"
        if not self.graph.has_node(entity_id):
            return []
        
        related = []
        for _, target, data in self.graph.edges(entity_id, data=True):
            if data['type'] == 'RELATES_TO' and data['weight'] >= min_weight:
                target_data = self.graph.nodes[target]
                related.append({
                    'entity_type': target_data['entity_type'],
                    'text': target_data['text'],
                    'weight': data['weight']
                })
        
        # Sort by weight descending
        return sorted(related, key=lambda x: x['weight'], reverse=True)
    
    def find_similar_documents(self, document_id: str, 
                             min_shared_entities: int = 2) -> List[Dict[str, Any]]:
        """Find documents similar to a given document based on shared entities."""
        if not self.graph.has_node(document_id):
            return []
        
        similar = []
        for _, target, data in self.graph.edges(document_id, data=True):
            if data['type'] == 'SIMILAR_TO' and data['weight'] >= min_shared_entities:
                target_data = self.graph.nodes[target]
                similar.append({
                    'document_id': target,
                    'content': target_data['content'],
                    'metadata': target_data['metadata'],
                    'shared_entities': data['weight']
                })
        
        # Sort by number of shared entities descending
        return sorted(similar, key=lambda x: x['shared_entities'], reverse=True)
    
    def trace_document_hierarchy(self, document_id: str) -> Dict[str, Any]:
        """Trace the document hierarchy (PART_OF relationships) for a document."""
        if not self.graph.has_node(document_id):
            return {}
        
        # Get document metadata
        doc_data = self.graph.nodes[document_id]
        
        # Find parts (outgoing PART_OF edges)
        parts = []
        for _, target, data in self.graph.edges(document_id, data=True):
            if data['type'] == 'PART_OF':
                parts.append(self.trace_document_hierarchy(target))
        
        return {
            'document_id': document_id,
            'metadata': doc_data['metadata'],
            'parts': parts
        }
    
    def multi_hop_query(self, start_entity: Dict[str, str], 
                       path_pattern: List[str]) -> List[Dict[str, Any]]:
        """
        Perform a multi-hop query following a specific path pattern.
        
        Args:
            start_entity: Dict with 'type' and 'text' for starting entity
            path_pattern: List of edge types to follow
            
        Example:
            path_pattern = ['CONTAINS', 'SIMILAR_TO', 'CONTAINS']
            This would find entities in similar documents to those containing the start entity
        """
        entity_id = f"{start_entity['type']}_{start_entity['text']}"
        if not self.graph.has_node(entity_id):
            return []
        
        # Start with the initial entity
        current_nodes = [entity_id]
        results = []
        
        # Follow each hop in the path pattern
        for edge_type in path_pattern:
            next_nodes = []
            for node in current_nodes:
                # Get all edges of the specified type
                if edge_type in ['CONTAINS', 'PART_OF', 'SIMILAR_TO']:
                    edges = self.graph.edges(node, data=True)
                else:  # For reverse traversal
                    edges = self.graph.in_edges(node, data=True)
                
                # Filter edges by type and collect target nodes
                for src, tgt, data in edges:
                    if data['type'] == edge_type:
                        next_node = tgt if src == node else src
                        next_nodes.append(next_node)
            
            current_nodes = list(set(next_nodes))  # Remove duplicates
            
            # If we've reached a dead end, stop
            if not current_nodes:
                break
        
        # Collect results from final nodes
        for node in current_nodes:
            node_data = self.graph.nodes[node]
            results.append({
                'node_id': node,
                'type': node_data['type'],
                'data': node_data
            })
        
        return results

# Load the graph from the previous notebook
# This assumes you've saved the graph or are running this in the same session
# If loading from a file, you would do something like:
# with open('graph.json', 'r') as f:
#     graph_data = json.load(f)
#     graph = nx.node_link_graph(graph_data)

# Initialize the query engine
query_engine = GraphQueryEngine(graph)

# Example queries
print("Documents containing 'AWS':")
aws_docs = query_engine.find_documents_with_entity('CLOUD_SERVICE', 'AWS')
for doc in aws_docs[:3]:  # Show first 3 results
    print(f"\nDocument: {doc['metadata']['source']}")
    print(f"Content preview: {doc['content'][:100]}...")

print("\nEntities related to 'Python':")
python_related = query_engine.find_related_entities('PROGRAMMING_LANGUAGE', 'Python')
for entity in python_related:
    print(f"- {entity['text']} (weight: {entity['weight']})")

print("\nMulti-hop query example:")
print("Finding entities in documents similar to those containing 'Neptune'")
results = query_engine.multi_hop_query(
    {'type': 'DATABASE', 'text': 'Neptune'},
    ['CONTAINS', 'SIMILAR_TO', 'CONTAINS']
)
